In [2]:
import pandas as pd
import numpy as np

# Criando o arquivo varejo.csv automaticamente na sessão do Colab
np.random.seed(42)
n = 300
df_simulado = pd.DataFrame({
    'id_cliente': [f"C{i:03d}" for i in range(1, n+1)],
    'nome': np.random.choice(['Ana Silva', 'carlos souza', '  Maria Oliveira ', 'JOAO PEREIRA', None, '   '], n),
    'idade': np.random.choice([25, 30, 35, 40, 45, None, 150], n),
    'genero': np.random.choice(['Feminino', 'Masculino', 'F', 'M', None], n),
    'categoria': np.random.choice(['Eletrônicos', 'Vestuário', 'Alimentação', 'Beleza', None], n),
    'valor_compra': np.random.choice([150.0, 45.5, 300.0, 1200.0, 89.9, None, -10.0], n),
    'quantidade': np.random.choice([1, 2, 3, 4, None], n)
})
df_simulado = pd.concat([df_simulado, df_simulado.iloc[:10]], ignore_index=True) # Adiciona duplicatas para o diagnóstico
df_simulado.to_csv('varejo.csv', index=False)

# Leitura para confirmar
df_original = pd.read_csv('varejo.csv')
print("Arquivo 'varejo.csv' criado e lido com sucesso!")
print(f"Dimensões: {df_original.shape[0]} linhas e {df_original.shape[1]} colunas.")
display(df_original.head())

Arquivo 'varejo.csv' criado e lido com sucesso!
Dimensões: 310 linhas e 7 colunas.


,id_cliente,nome,idade,genero,categoria,valor_compra,quantidade
0,C001,JOAO PEREIRA,NaN,NaN,Eletrônicos,NaN,1.0
1,C002,NaN,25.0,Masculino,Beleza,-10.0,3.0
2,C003,Maria Oliveira,35.0,NaN,Vestuário,300.0,2.0
3,C004,NaN,25.0,F,Vestuário,300.0,3.0
4,C005,NaN,45.0,NaN,Eletrônicos,300.0,NaN


In [3]:
import pandas as pd

# Leitura do CSV
df_original = pd.read_csv('varejo.csv')

# Apresentar as primeiras linhas
print("--- PRIMEIRAS LINHAS ---")
display(df_original.head())

# Quantidade de registros e colunas
print(f"\nDimensões da base: {df_original.shape[0]} linhas e {df_original.shape[1]} colunas.")

# Nomes das colunas e tipos de dados
print("\nNomes e Tipos de Dados:")
print(df_original.dtypes)

# Diagnóstico de valores ausentes (nulos)
print("\nValores ausentes por coluna:")
print(df_original.isnull().sum())

# Linhas duplicadas
print(f"\nLinhas duplicadas confirmadas: {df_original.duplicated().sum()}")

# Textos vazios ou espaços na coluna 'nome'
vazios = df_original['nome'].astype(str).str.strip().eq("").sum()
print(f"Textos vazios ou só com espaços na coluna 'nome': {vazios}")

--- PRIMEIRAS LINHAS ---


,id_cliente,nome,idade,genero,categoria,valor_compra,quantidade
0,C001,JOAO PEREIRA,NaN,NaN,Eletrônicos,NaN,1.0
1,C002,NaN,25.0,Masculino,Beleza,-10.0,3.0
2,C003,Maria Oliveira,35.0,NaN,Vestuário,300.0,2.0
3,C004,NaN,25.0,F,Vestuário,300.0,3.0
4,C005,NaN,45.0,NaN,Eletrônicos,300.0,NaN



Dimensões da base: 310 linhas e 7 colunas.

Nomes e Tipos de Dados:
id_cliente       object
nome             object
idade           float64
genero           object
categoria        object
valor_compra    float64
quantidade      float64
dtype: object

Valores ausentes por coluna:
id_cliente       0
nome            43
idade           48
genero          80
categoria       58
valor_compra    45
quantidade      68
dtype: int64

Linhas duplicadas confirmadas: 10
Textos vazios ou só com espaços na coluna 'nome': 46


In [4]:
def tratar_dados_varejo(df):
    """Função de tratamento com parâmetro e retorno"""
    df_tratado = df.copy()

    # Remover apenas duplicatas exatas
    df_tratado = df_tratado.drop_duplicates()

    # Padronizar texto da coluna 'nome'
    if 'nome' in df_tratado.columns:
        df_tratado['nome'] = df_tratado['nome'].fillna("Cliente Desconhecido")
        df_tratado['nome'] = df_tratado['nome'].astype(str).str.strip().str.title()
        df_tratado.loc[df_tratado['nome'] == "", 'nome'] = "Cliente Desconhecido"

    # Padronizar gênero
    if 'genero' in df_tratado.columns:
        df_tratado['genero'] = df_tratado['genero'].fillna("Não Informado")
        df_tratado['genero'] = df_tratado['genero'].replace({"F": "Feminino", "M": "Masculino"})

    # Tratar categoria
    if 'categoria' in df_tratado.columns:
        df_tratado['categoria'] = df_tratado['categoria'].fillna("Outros")

    # Tratar valores numéricos de compra
    if 'valor_compra' in df_tratado.columns:
        df_tratado['valor_compra'] = pd.to_numeric(df_tratado['valor_compra'], errors='coerce')
        mediana = df_tratado['valor_compra'].median()
        df_tratado['valor_compra'] = df_tratado['valor_compra'].fillna(mediana).abs()

    return df_tratado

def gerar_resumo_estatistico(df):
    """Função de resumo estatístico com parâmetro e retorno"""
    resumo = {
        "total_registros": len(df),
        "faturamento_total": float(df['valor_compra'].sum()),
        "ticket_medio": float(df['valor_compra'].mean())
    }
    return resumo

# Aplicando as funções
df_limpo = tratar_dados_varejo(df_original)
resumo_executivo = gerar_resumo_estatistico(df_limpo)

print(f"Registros antes da limpeza: {len(df_original)}")
print(f"Registros após a limpeza: {len(df_limpo)}")
print("\nResumo Executivo Gerado:")
print(resumo_executivo)

Registros antes da limpeza: 310
Registros após a limpeza: 300

Resumo Executivo Gerado:
{'total_registros': 300, 'faturamento_total': 69111.00000000001, 'ticket_medio': 230.37000000000006}


In [5]:
coluna = 'valor_compra'

# Estatísticas básicas obrigatórias
print(f"Estatísticas da coluna '{coluna}':")
print(f" - Contagem válida: {df_limpo[coluna].count()}")
print(f" - Média (Ticket Médio): R$ {df_limpo[coluna].mean():.2f}")
print(f" - Mínimo: R$ {df_limpo[coluna].min():.2f}")
print(f" - Máximo: R$ {df_limpo[coluna].max():.2f}")

# Filtro aplicado (ex: compras acima de 100 reais)
df_filtrado = df_limpo[df_limpo['valor_compra'] > 100.00]
print(f"\nRegistros com valor superior a R$ 100,00: {len(df_filtrado)}")

# Agrupamento 1: Contagem por Categoria
print("\n--- Agrupamento 1: Compras por Categoria ---")
agrp_cat = df_limpo.groupby('categoria')['id_cliente'].count().reset_index()
agrp_cat.columns = ['Categoria', 'Quantidade de Compras']
print(agrp_cat)

# Agrupamento 2: Soma por Gênero
print("\n--- Agrupamento 2: Faturamento por Gênero ---")
agrp_gen = df_limpo.groupby('genero')['valor_compra'].sum().reset_index()
agrp_gen.columns = ['Gênero', 'Faturamento Total (R$)']
print(agrp_gen)

Estatísticas da coluna 'valor_compra':
 - Contagem válida: 300
 - Média (Ticket Médio): R$ 230.37
 - Mínimo: R$ 10.00
 - Máximo: R$ 1200.00

Registros com valor superior a R$ 100,00: 122

--- Agrupamento 1: Compras por Categoria ---
     Categoria  Quantidade de Compras
0  Alimentação                     61
1       Beleza                     58
2  Eletrônicos                     69
3       Outros                     57
4    Vestuário                     55

--- Agrupamento 2: Faturamento por Gênero ---
          Gênero  Faturamento Total (R$)
0       Feminino                 25663.3
1      Masculino                 24913.9
2  Não Informado                 18533.8


In [6]:
# Exportar base limpa
arquivo_saida = "varejo_limpo.csv"
df_limpo.to_csv(arquivo_saida, index=False)

print(f"Base tratada exportada com sucesso para '{arquivo_saida}'!")

Base tratada exportada com sucesso para 'varejo_limpo.csv'!
